In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("./final_df.csv")
df.sample(3)

,question,evidence,ground_truth,url,source
1,Is the activity graph an abstraction of the MO...,['The key abstraction of MOMA-LRG is the activ...,Yes.,https://openreview.net/forum?id=eJhc_CPXQIT,3
12,How does the arborescence-based training model...,"[""Graph-based Dissimilarity Let Gbe a graph wi...",The model calculates dissimilarity using a min...,https://aclanthology.org/2022.naacl-main.343.pdf,1
3,How can the authors ensure that the natural la...,"['To do this , we provide graphical annotation...",We ensure this by making sure that the natural...,https://openreview.net/forum?id=eJhc_CPXQIT,3


In [3]:
from main import run_pipeline
from datetime import datetime

db_dir = {
    "1": "./outputs/test1/",
    "2": "./outputs/test2/",
    "3": "./outputs/test3/"
}

In [41]:
start = datetime.now()
all_answers = []

for db_id, input_dir in db_dir.items():
    print(f"Running pipeline for DB {db_id}...")
    rows = df[df["source"] == int(db_id)]

    for _, row in rows.iterrows():
        question = row["question"]
        print(f"Question: {question}")
        answer, retrieved_content, decision = run_pipeline(query=question, vector_db_path=input_dir, top_k=3)
    
        all_answers.append({
                "question": question,
                "answer": answer,
                "context": retrieved_content,
                "ground_truth": row["ground_truth"],
                "evidence": row["evidence"],
                "source": db_id
            })
end = datetime.now()
ellapsed_time = end - start
print(f"Total time taken: {ellapsed_time}")

Running pipeline for DB 1...
Question: What is the target KB size used for MedMentions in the experiments?
Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Tree PNG saved -> decompositions\test\decomposition_Do_the_training_and__fcbca259-58df-47c0-9a24-37cd23919438.png
Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Tree PNG saved -> decompositions\test\decomposition_Does_the_model_need__2b6e3a15-4c5d-45ce-880d-473bcd15a5b6.png
Question: Why not use the complete graph for training instead of pruning it?
Tree PNG saved -> decompositions\test\decomposition_Why_not_use_the_comp_894dbb9c-f9c5-4f98-b34e-914f0510733c.png
Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for this calculation?
Tree PNG saved

In [50]:
print(f"{(ellapsed_time/95).seconds} seconds per question on average")

21 seconds per question on average


In [56]:
((ellapsed_time/95) * 7).seconds

149

In [55]:
((ellapsed_time/95) * 7).seconds / 60

2.4833333333333334

In [43]:
all_answers

[{'question': 'What is the target KB size used for MedMentions in the experiments?',
  'answer': '2.3 million entities (2.3M)',
  'context': [Document(id='bea9f360-00ea-4140-a561-aa4e8c69ec58', metadata={'source': 'inputs\\test1\\test1.txt'}, page_content='Table 2: Dataset Statistics. is the number of mentions.  is the number of unique entities in the labeled partition (not the total KB size). \\(| \\mathcal { E } \\backslash \\mathcal { E } _ { \\mathrm { T r a i n } } |\\) is the number of zero-shot entities. The total KB size of MedMentions and ZeShEL is 2.3M and 492K, respectively.   \n\n<table><tr><td></td><td></td><td>MedMentions</td><td>ZeShEL</td></tr><tr><td rowspan="3">|M|</td><td>Train</td><td>120K</td><td>49K</td></tr><tr><td>Dev</td><td>40K</td><td>10K</td></tr><tr><td>Test</td><td>40K</td><td>10K</td></tr><tr><td rowspan="3">|E|</td><td>Train</td><td>19K</td><td>26K</td></tr><tr><td>Dev</td><td>9K</td><td>7K</td></tr><tr><td>Test</td><td>8K</td><td>7K</td></tr><tr><td row

In [44]:
len(all_answers)

19

In [45]:
all_answers_df = pd.DataFrame(all_answers)
all_answers_df.to_csv("all_answers.csv", index=False)

In [46]:
all_answers_df.sample(2)

,question,answer,context,ground_truth,evidence,source
2,"Does the model need to compute \psi(m_i, e) fo...",No. During inference you should not compute ps...,[page_content='and the weight of the edge from...,No.,"['For each mentionm , construct Em by ( a ) ad...",1
9,What BLEU score did the baseline model develop...,- Vaswani et al.'s baseline BLEU on WMT14 EN-D...,[page_content='We choose the standard WMT 2014...,"The Transformer base model achieved 27.3 BLEU,...","[""Table 2: The Transformer achieves better BLE...",2


In [16]:
def f1_token_overlap(pred, truth):
    pred_tokens = set(pred.lower().split())
    truth_tokens = set(truth.lower().split())
    if not pred_tokens or not truth_tokens:
        return 0
    return 2 * len(pred_tokens & truth_tokens) / (len(pred_tokens) + len(truth_tokens))

all_answers_df['f1'] = all_answers_df.apply(
    lambda row: f1_token_overlap(row['answer'], row['ground_truth']), axis=1
)

In [17]:
all_answers_df.sample(2)

,question,answer,context,ground_truth,evidence,source,f1
2,Is the dataset under a Creative Commons license?,"Not necessarily. Licenses vary by dataset, and...","[page_content='[1] Jia Deng, Wei Dong, Richard...",Yes.,"['Specifically , we will use Attribution 4.0 I...",3,0.000000
1,What specific issue with the dot-product atten...,- Issue at large dk: The dot products QK^T can...,[page_content='\[\n\boldsymbol {q} _ {m} ^ {\i...,Noam Shazeer and his co-authors suspected that...,"[""While for small values of dkthe two mechanis...",2,0.338462


In [47]:
from rag_eval import evaluate

results = []
for row in all_answers_df.itertuples():
    print(f"Evaluating Question: {row.question}")
    result = evaluate(
        question=row.question,
        answer=row.answer,
        contexts=[doc.page_content for doc in row.context],
        ground_truth=row.ground_truth
    )
    results.append(result)

Evaluating Question: What is the target KB size used for MedMentions in the experiments?
Evaluating Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Evaluating Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Evaluating Question: Why not use the complete graph for training instead of pruning it?
Evaluating Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for this calculation?
Evaluating Question: Why do the proposed arborescence-based cross-encoder models show consistent gains in linking accuracy on the MedMentions dataset but mixed results compared to mention-entity baselines on the ZeShEL dataset, and what specific statistical difference accounts for this?
Evaluating Question: During the constraine

In [48]:
df_with_eval = all_answers_df.copy()

In [49]:
for i, result in enumerate(results):
    for metric_name, metric_value in result.items():
        if isinstance(metric_value, dict):
            df_with_eval.at[i, metric_name] = metric_value.get("score")
        else:
            df_with_eval.at[i, metric_name] = metric_value

In [36]:
df_with_eval

,question,answer,context,ground_truth,evidence,source,f1,faithfulness,context_relevance,answer_relevance,answer_correctness,overall
0,"In the Oracle Inference experiment, how do the...",- How SELF vs UNION differ:\n - SELF: A per-r...,[page_content='#### 3.4 Oracle Inference\n\nIn...,"The ""SELF"" setting isolates re-ranking capabil...","[""Finally, we also provide representative exam...",1,0.320388,1.0,1.0,1.0,1.0,1.000
1,What specific issue with the dot-product atten...,- Issue at large dk: The dot products QK^T can...,[page_content='\[\n\boldsymbol {q} _ {m} ^ {\i...,Noam Shazeer and his co-authors suspected that...,"[""While for small values of dkthe two mechanis...",2,0.338462,1.0,1.0,1.0,1.0,1.000
2,Is the dataset under a Creative Commons license?,"Not necessarily. Licenses vary by dataset, and...","[page_content='[1] Jia Deng, Wei Dong, Richard...",Yes.,"['Specifically , we will use Attribution 4.0 I...",3,0.000000,1.0,0.0,1.0,0.5,0.625


In [60]:
df_with_eval.to_csv("all_answers_with_eval.csv", index=False)

In [61]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          1.000
context_relevance     0.850
answer_relevance      0.975
answer_correctness    0.875
overall               0.925
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.833333
context_relevance     0.600000
answer_relevance      0.900000
answer_correctness    0.833333
overall               0.791667
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          0.9500
context_relevance     0.6000
answer_relevance      0.9875
answer_correctness    0.5625
overall               0.7750
dtype: float64
